In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import pandas as pd
customers = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
customers.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
customers.shape

(7043, 21)

In [5]:
customers.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

## Initial Data Exploration

The dataset contains 7,043 customers and 21 features, including the target 
variable 'Churn'.

**Class imbalance observed**: Only 26.5% of customers churned (1,869 out of 7,043). 
This means accuracy alone will be a misleading metric. A model that predicts 
"No churn" for every customer would still be 73.5% accurate. We'll need to evaluate. 
using precision, recall, and F1-score instead.

**Data quality flag:** 'TotalCharges' is stored as object dtype despite appearing 
numeric. Investigating further...

In [6]:
customers.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
customers['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [8]:
customers.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [9]:
customers['TotalCharges'].unique()[:20]

array(['29.85', '1889.5', '108.15', '1840.75', '151.65', '820.5',
       '1949.4', '301.9', '3046.05', '3487.95', '587.45', '326.8',
       '5681.1', '5036.3', '2686.05', '7895.15', '1022.95', '7382.25',
       '528.35', '1862.9'], dtype=object)

In [10]:
customers[customers['TotalCharges'] == ' ']['TotalCharges'].count()

np.int64(11)

In [11]:
customers['TotalCharges'] = customers['TotalCharges'].replace(' ', np.nan)
customers['TotalCharges'] = customers['TotalCharges'] .astype('float')
customers['TotalCharges'].isnull().sum()

np.int64(11)

In [12]:
customers[customers['TotalCharges'].isnull()][["tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


## Data Quality Fix — TotalCharges Column

'TotalCharges' contained 11 rows with whitespace '" "' instead of numeric values. These were replaced with 'NaN' and the column was converted to 'float.'

**Root cause:** These 11 customers have 'tenure = 0,' meaning they just joined and 
have no charges yet. This is a data pipeline issue; new customers were added before their first bill was generated.

**Decision:** Drop these 11 rows (0.15% of data) as they represent customers with 
no billing history and no meaningful churn signal. 

In [13]:
df = customers.dropna()
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [14]:
df.shape

(7032, 21)

## Handling Missing Values
Dropped 11 rows (0.15% of dataset). A customer with zero tenure has no behavioral signal for churn prediction. Keeping them would introduce noise.

Clean dataset of 7,032 customers ready for analysis.

In [15]:
cat_cols = df.select_dtypes(include='object').columns.to_list()
print(cat_cols)

['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


In [16]:
for col in cat_cols:
    print(f"\n{col} : {df[col].unique()}")


customerID : ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']

gender : ['Female' 'Male']

Partner : ['Yes' 'No']

Dependents : ['No' 'Yes']

PhoneService : ['No' 'Yes']

MultipleLines : ['No phone service' 'No' 'Yes']

InternetService : ['DSL' 'Fiber optic' 'No']

OnlineSecurity : ['No' 'Yes' 'No internet service']

OnlineBackup : ['Yes' 'No' 'No internet service']

DeviceProtection : ['No' 'Yes' 'No internet service']

TechSupport : ['No' 'Yes' 'No internet service']

StreamingTV : ['No' 'Yes' 'No internet service']

StreamingMovies : ['No' 'Yes' 'No internet service']

Contract : ['Month-to-month' 'One year' 'Two year']

PaperlessBilling : ['Yes' 'No']

PaymentMethod : ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Churn : ['No' 'Yes']


In [17]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [18]:
# Currently 0/1 — convert to Yes/No for consistency
df.loc[df['SeniorCitizen'] == 'Yes', ['SeniorCitizen']] = 1
df.loc[df['SeniorCitizen'] == 'No', ['SeniorCitizen']] = 0

df['SeniorCitizen'].value_counts()

SeniorCitizen
0    5890
1    1142
Name: count, dtype: int64

In [19]:
#Replace 'No internet service' with 'No' across all 6 columns
no_internet_cols = ['StreamingMovies', 'StreamingTV', 'TechSupport', 'DeviceProtection', 'OnlineSecurity', 'OnlineBackup']

for col in no_internet_cols:
    df.loc[df[col] == 'No internet service', [col]] = 'No'
    
# Same for MultipleLines
df.loc[df['MultipleLines'] == 'No phone service', ['MultipleLines']] = 'No'

# Verify
for col in no_internet_cols:
    print(f"\n{col} : {df[col].unique()}")


StreamingMovies : ['No' 'Yes']

StreamingTV : ['No' 'Yes']

TechSupport : ['No' 'Yes']

DeviceProtection : ['No' 'Yes']

OnlineSecurity : ['No' 'Yes']

OnlineBackup : ['Yes' 'No']


In [20]:
binary_cols = ['StreamingMovies', 'StreamingTV', 'TechSupport', 'DeviceProtection', 'OnlineSecurity', 'OnlineBackup', 'MultipleLines', 'SeniorCitizen', 'PaperlessBilling', 'PhoneService', 'Dependents', 'Partner', 'gender']

for col in binary_cols:
    df.loc[df[col] == 'Yes', [col]] = 1
    df.loc[df[col] == 'No', [col]] = 0
    df.loc[df[col] == 'Female', [col]] = 1
    df.loc[df[col] == 'Male', [col]] = 0

for col in no_internet_cols:
    print(f"\n{col} : {df[col].unique()}")
    
# Encode target variable

df.loc[df['Churn'] == 'Yes', ['Churn']] = 1
df.loc[df['Churn'] == 'No', ['Churn']] = 0

df['Churn'].value_counts()



StreamingMovies : [0 1]

StreamingTV : [0 1]

TechSupport : [0 1]

DeviceProtection : [0 1]

OnlineSecurity : [0 1]

OnlineBackup : [1 0]


Churn
0    5163
1    1869
Name: count, dtype: int64

In [21]:
other_cols = ['PaymentMethod', 'Contract', 'InternetService']

new_df = pd.get_dummies(df, columns=other_cols, drop_first=False)

new_df.shape

(7032, 28)

In [22]:
new_df.columns.to_list()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'PaymentMethod_Bank transfer (automatic)',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No']

In [23]:
new_df.shape

(7032, 28)

In [24]:
new_df.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No
0,7590-VHVEG,1,0,1,0,1,0,0,0,1,...,False,False,True,False,True,False,False,True,False,False
1,5575-GNVDE,0,0,0,0,34,1,0,1,0,...,False,False,False,True,False,True,False,True,False,False
2,3668-QPYBK,0,0,0,0,2,1,0,1,1,...,False,False,False,True,True,False,False,True,False,False
3,7795-CFOCW,0,0,0,0,45,0,0,1,0,...,True,False,False,False,False,True,False,True,False,False
4,9237-HQITU,1,0,0,0,2,1,0,0,0,...,False,False,True,False,True,False,False,False,True,False


In [25]:
new_df = new_df.rename(columns={
    # InternetService
    'InternetService_DSL' : 'internet_service_dsl',
    'InternetService_Fiber optic' : 'internet_service_fiber',
    'InternetService_No' : 'internet_service_none',
    
    # Contract
    'Contract_Month-to-month' : 'contract_monthly',
    'Contract_One year' : 'contract_one_year',
    'Contract_Two year' : 'contract_two_year',

    # PaymentMethod
    'PaymentMethod_Bank transfer (automatic)' : 'payment_bank_transfer',
    'PaymentMethod_Credit card (automatic)' : 'payment_credit_card',
    'PaymentMethod_Electronic check' : 'payment_electronic_check',
    'PaymentMethod_Mailed check' : 'payment_mailed_check'
})

new_df.columns.tolist()

['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'payment_bank_transfer',
 'payment_credit_card',
 'payment_electronic_check',
 'payment_mailed_check',
 'contract_monthly',
 'contract_one_year',
 'contract_two_year',
 'internet_service_dsl',
 'internet_service_fiber',
 'internet_service_none']

In [26]:
new_df = new_df.drop('customerID', axis=1)
new_df.shape

(7032, 27)

In [27]:
new_df.shape

(7032, 27)

In [28]:
# Feature 1 - Charges relative to how long they've stayed
new_df['charges_per_tenure'] = new_df['MonthlyCharges']/(new_df['tenure'] + 1)

# Feature 2 - Whether customer is locked into a long-term contract
new_df['is_long_term_contract'] = ((new_df['contract_one_year'] == 1) | (new_df['contract_two_year'] == 1)).astype(int)

new_df[['charges_per_tenure', 'is_long_term_contract']].describe()


,charges_per_tenure,is_long_term_contract
count,7032.000000,7032.000000
mean,5.714882,0.448948
std,8.567435,0.497422
min,0.264384,0.000000
25%,1.250000,0.000000
50%,2.073598,0.000000
75%,5.884842,1.000000
max,51.225000,1.000000


## Feature Engineering

Two new features were created:

**'charges_per_tenure'** - Monthly charges divided by tenure. Captures whether a customer is paying a lot relative to how long they've been around. High values indicate recent customers with expensive plans - a strong churn risk signal.

**'is_long_term_contract'** - Binary flag for customers on 1- or 2-year contracts. Encodes switching friction directly. Customers on month-to-month contracts (55% of the dataset) have no commitment barrier and are significantly more likely to churn.

## Outlier Analysis

'charges_per_tenure' shows a max of 51.2 against a mean of 5.7, indicating 
right-skewed distribution with extreme values.

**Decision:** Outliers retained intentionally. This project uses Random Forest which is robust to outliers by design. Additionally, high charges_per_tenure values represent genuine high-risk customers - capping them would remove a 
meaningful churn signal.

For Logistic Regression, StandardScaler applied during preprocessing will reduce the influence of extreme values sufficiently.

In [29]:
#Target variable
y = new_df['Churn']
# Feature matrix — everything except Churn
X = new_df.drop('Churn', axis=1)
print("X shape", X.shape)
print("y shape", y.shape)
print("Churn rate", round(y.mean()*100, 2), '%')

X shape (7032, 28)
y shape (7032,)
Churn rate 26.58 %


In [30]:
new_df.to_csv("telco_churn_clean.csv", index=False)